# Carnage-V1 on Colab (v3-style)

1v1 Rocket League self-play trainer (GigaLearnCPP + RLGymCPP), Phase-1 reward stack, GPU training.

**How to run:**
1. Ensure **Runtime -> Change runtime type -> T4 GPU** is selected, then restart.
2. Run cells top to bottom.
3. Cell 2 mounts your Drive (click the auth link once). Checkpoints auto-backup to `My Drive/Carnage-V1/checkpoints/` from the start.
4. Optional: upload the converted replay binary to `My Drive/Carnage-V1/serialized_replays.bin`.
5. Use the last cell (STOP) to stop training cleanly - it saves a checkpoint first.

Source: `https://github.com/vfxjamer/Carnage-V1.git`

In [1]:
# cell - 1
# 1. Clone source (self-contained: src + CMakeLists + collision_meshes + GigaLearnCPP thirdparty)
import os, subprocess, shutil

ROOT = "/content/Carnage-V1"
REPO = "https://github.com/vfxjamer/Carnage-V1.git"

if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "pull"], check=False)

print("ROOT:", ROOT)
print("contents:", sorted(os.listdir(ROOT)))

ROOT: /content/Carnage-V1
contents: ['.git', '.gitignore', 'CMakeLists.txt', 'Carnage_colab.ipynb', 'colab_setup.sh', 'collision_meshes', 'src', 'thirdparty']


In [ ]:
# cell - 2
# 2. Mount Google Drive EARLY (so checkpoints back up from the start) + restore latest checkpoint.
import os, shutil, glob, time

def drive_is_healthy():
    try:
        return (
            os.path.isdir("/content/drive/MyDrive")
            and os.listdir("/content/drive/MyDrive") is not None
        )
    except OSError:
        return False

if not drive_is_healthy():
    print("mounting/remounting Drive (complete the auth popup in the browser)...", flush=True)
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print("WARN: drive mount failed:", e, "- backups will retry automatically", flush=True)
else:
    print("Drive is healthy", flush=True)

DRIVE_CKPT = "/content/drive/MyDrive/Carnage-V1/checkpoints"
LOCAL_CKPT = "/content/Carnage-V1/build/checkpoints"
DRIVE_REPLAY = "/content/drive/MyDrive/Carnage-V1/serialized_replays.bin"
LOCAL_REPLAY = "/content/Carnage-V1/build/serialized_replays.bin"
PARTIAL_REPLAY = LOCAL_REPLAY + ".part"

os.makedirs(LOCAL_CKPT, exist_ok=True)

def ts_of(d):
    try:
        return int(os.path.basename(d))
    except Exception:
        return -1

if os.path.isdir(DRIVE_CKPT):
    drive_dirs = sorted(
        [
            d for d in glob.glob(os.path.join(DRIVE_CKPT, "*"))
            if os.path.isdir(d)
        ],
        key=ts_of
    )

    print(f"Drive has {len(drive_dirs)} checkpoint dirs")

    if drive_dirs:
        newest = drive_dirs[-1]
        dest = os.path.join(LOCAL_CKPT, os.path.basename(newest))

        if not os.path.isdir(dest):
            print("restoring latest checkpoint:", os.path.basename(newest))
            shutil.copytree(newest, dest)
else:
    print("no Drive checkpoints yet")

# Copy replay safely: partial files never become the active replay file.
if os.path.exists(DRIVE_REPLAY):

    drive_replay_size = os.path.getsize(DRIVE_REPLAY)

    # If a local replay already exists, make sure it is complete.
    if os.path.isfile(LOCAL_REPLAY):
        local_replay_size = os.path.getsize(LOCAL_REPLAY)

        if local_replay_size == drive_replay_size:
            print(
                f"replay binary already present and verified "
                f"({local_replay_size:,} bytes)"
            )
        else:
            print(
                f"local replay size mismatch "
                f"({local_replay_size:,} != {drive_replay_size:,}); recopying"
            )
            os.remove(LOCAL_REPLAY)

    if not os.path.exists(LOCAL_REPLAY):

        print("copying replay binary from Drive")

        for attempt in range(3):
            try:
                # Remove any incomplete transfer from a previous attempt.
                if os.path.exists(PARTIAL_REPLAY):
                    os.remove(PARTIAL_REPLAY)

                print(
                    f"replay copy attempt {attempt + 1}/3 "
                    f"({drive_replay_size:,} bytes)"
                )

                # Copy into .part first.
                shutil.copy2(DRIVE_REPLAY, PARTIAL_REPLAY)

                # Verify the transfer before exposing it as the real replay.
                copied_size = os.path.getsize(PARTIAL_REPLAY)

                if copied_size != drive_replay_size:
                    raise OSError(
                        f"replay copy size mismatch: "
                        f"{copied_size:,} != {drive_replay_size:,} bytes"
                    )

                # Atomic rename: only a verified replay becomes active.
                os.replace(PARTIAL_REPLAY, LOCAL_REPLAY)

                print("replay binary copied and verified successfully")
                break

            except OSError as e:
                print(
                    f"replay copy failed "
                    f"(attempt {attempt + 1}/3): {e}",
                    flush=True
                )

                # Never leave a broken partial file behind.
                try:
                    if os.path.exists(PARTIAL_REPLAY):
                        os.remove(PARTIAL_REPLAY)
                except OSError:
                    pass

                if attempt == 2:
                    raise

                print("remounting Google Drive...", flush=True)

                try:
                    from google.colab import drive
                    drive.mount("/content/drive", force_remount=True)
                except Exception as mount_err:
                    print("Drive remount failed:", mount_err)

                time.sleep(5)

else:
    print("WARNING: replay binary not found on Drive:", DRIVE_REPLAY)

print(
    "local checkpoints:",
    sorted(os.listdir(LOCAL_CKPT))
    if os.path.isdir(LOCAL_CKPT)
    else []
)

print(
    "replay binary present:",
    os.path.exists(LOCAL_REPLAY)
)

if os.path.exists(LOCAL_REPLAY):
    print(
        "replay binary size:",
        f"{os.path.getsize(LOCAL_REPLAY):,}",
        "bytes"
    )

mounting/remounting Drive (complete the auth popup in the browser)...
Mounted at /content/drive
Drive has 1 checkpoint dirs
restoring latest checkpoint: 355027456


In [ ]:
# cell - 3
# 3. Sanity check: CLI flags, collision meshes, PPO config markers in the cloned source.
import os
ROOT = "/content/Carnage-V1"

def _has(path, needle, nice):
    if not os.path.exists(path):
        print("MISS:", path); return
    s = open(path).read()
    ok = needle in s
    print(("OK  " if ok else "MISSING ") + nice, "<-", os.path.relpath(path, ROOT))

main = os.path.join(ROOT, "src", "main.cpp")
_has(main, "--device", "cli --device")
_has(main, "--games", "cli --games")
_has(main, "--save-dir", "cli --save-dir")
_has(main, "--wandb", "cli --wandb")
_has(main, "NextoObs", "Nexto obs builder")
_has(main, "LayerNorm", "LayerNorm config")

meshes = os.path.join(ROOT, "collision_meshes")
print("collision_meshes dir:", os.path.isdir(meshes), "|", sorted(os.listdir(meshes)) if os.path.isdir(meshes) else "")

print("Carnage sanity check done.")

In [ ]:
# cell - 4a
# Install numpy 1.26.4 (required for wandb 0.16.6 compatibility)
# AFTER THIS CELL: Runtime -> Restart session
import subprocess, sys
print("Installing numpy==1.26.4...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir", "numpy==1.26.4"], capture_output=True, text=True)
print("pip numpy rc:", r.returncode)
if r.returncode != 0:
    print(r.stderr[-500:])

import numpy
print("NumPy version:", numpy.__version__)
print("np.float_ available:", hasattr(numpy, "float_"))

print("\n>>> NOW: Runtime -> Restart session <<<")

In [ ]:
# cell - 4b
# Install wandb 0.16.6 (run AFTER restarting runtime)
import os, subprocess, sys
WANDB_API_KEY = "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"

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_MODE"] = "online"
os.environ["CARNAGE_WANDB_GROUP"] = "Phase 2"
os.environ["CARNAGE_WANDB_RUN"] = "carnage-v1"

print("clearing stale wandb modules...")
for _m in list(sys.modules):
    if _m == "wandb" or _m.startswith("wandb."):
        del sys.modules[_m]

print("Installing wandb==0.16.6...")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-cache-dir", "wandb==0.16.6"], capture_output=True, text=True)
print("pip wandb rc:", r.returncode, (r.stderr or "")[-200:])

import numpy
import wandb
print("NumPy:", numpy.__version__)
print("W&B:", wandb.__version__)
print("np.float_:", hasattr(numpy, "float_"))
print("key set:", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
# cell - 5
# 2. Install build deps (apt) + ensure GPU-enabled torch via pip if needed
import subprocess, sys, os

APT_PKGS = ["build-essential", "cmake", "git", "libpython3-dev", "pkg-config"]
print("apt update/install...")
r = subprocess.run(["apt-get", "update", "-qq"], capture_output=True, text=True)
print("update rc:", r.returncode, (r.stderr or "")[-300:])
r = subprocess.run(["apt-get", "install", "-y", "-qq"] + APT_PKGS, capture_output=True, text=True)
print("install rc:", r.returncode, (r.stderr or "")[-300:])

# torch: pip's libtorch headers are used by CMake (TORCH_INSTALL_PREFIX).
# On a GPU runtime ensure CUDA-enabled torch. On CPU runtime keep whatever is there.
import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available(), "cuda build:", torch.version.cuda)
if torch.version.cuda is None:
    print("NOTE: torch is CPU-only. Training cell will require switching to GPU runtime.")

cmake_v = subprocess.run(["cmake", "--version"], capture_output=True, text=True).stdout
print(cmake_v.splitlines()[0])
print("gcc:", subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
print("python dev headers:", os.path.exists("/usr/include/python3.12/Python.h") or os.path.exists("/usr/local/include/python3.12/Python.h") or os.path.exists("/usr/include/python3.11/Python.h"))

In [ ]:
# cell - 6
# 3. Configure + build Carnage (Release). Override TORCH_INSTALL_PREFIX to pip's torch location.
import os, subprocess, sys

ROOT = "/content/Carnage-V1"
os.chdir(ROOT)

import torch
torch_prefix = os.path.dirname(torch.__file__)  # contains share/cmake/Torch
print("TORCH_INSTALL_PREFIX =", torch_prefix)
print("has Torch cmake:", os.path.exists(os.path.join(torch_prefix, "share", "cmake", "Torch")))

configure = [
    "cmake", "-S", ".", "-B", "build",
    "-DCMAKE_BUILD_TYPE=Release",
    f"-DTORCH_INSTALL_PREFIX={torch_prefix}",
]
print(" ".join(configure))
r = subprocess.run(configure, capture_output=True, text=True)
print("configure rc:", r.returncode)
print((r.stdout or "")[-2500:])
print((r.stderr or "")[-1500:])

In [ ]:
# cell - 7
# 4. Build (this is the long step). Uses all available cores.
import os, subprocess

ROOT = "/content/Carnage-V1"
os.chdir(ROOT)

nproc = os.cpu_count() or 2
print(f"Building with -j{nproc} ... (this can take 10-30 min)")
r = subprocess.run(["cmake", "--build", "build", "-j", str(nproc)], capture_output=True, text=True)
print("build rc:", r.returncode)
tail = (r.stdout or "")[-3000:] + (r.stderr or "")[-2000:]
print(tail)
print("---")
exe = os.path.join(ROOT, "build", "Carnage")
print("binary exists:", os.path.exists(exe), exe if os.path.exists(exe) else "")

In [ ]:
# cell - 8
import sys, platform
print("python:", sys.version.split()[0])
print("platform:", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch import:", e)

In [ ]:
# cell - 9
# 4b. System performance monitor -> wandb (GPU util/VRAM/temp, CPU, RAM, disk I/O)
# Runs in the background while training. Uses the SAME wandb run via run_id from the binary log.
import os, time, threading, subprocess, glob, re

os.environ["WANDB_API_KEY"] = WANDB_API_KEY

import numpy as np
if not hasattr(np, "float_"):
    np.float_ = np.float64
if not hasattr(np, "complex_"):
    np.complex_ = np.complex128

def _read_run_id():
    logs = glob.glob("/content/Carnage-V1/build/*.log") + glob.glob("/content/Carnage-V1/*.log")
    for f in logs:
        try:
            txt = open(f, errors="ignore").read()
            m = re.search(r'run with ID : "([^"]+)"', txt)
            if m: return m.group(1)
        except Exception:
            pass
    return None

def _nvidia():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout.strip()
        vals = out.split(",")
        return dict(gpu_util=float(vals[0]), vram_used_mb=float(vals[1]), vram_total_mb=float(vals[2]), gpu_temp_c=float(vals[3]))
    except Exception:
        return None

def _cpu():
    try:
        import psutil
        return dict(cpu_util=psutil.cpu_percent(interval=1), ram_used_gb=round(psutil.virtual_memory().used / 1e9, 2),
                    ram_total_gb=round(psutil.virtual_memory().total / 1e9, 2))
    except Exception:
        return None

def sys_monitor(project, group):
    import wandb
    if not hasattr(np, "float_"):
        np.float_ = np.float64
    if not hasattr(np, "complex_"):
        np.complex_ = np.complex128
    while not os.path.exists("/content/Carnage-V1/build/Carnage"):
        time.sleep(10)
    run_id = None
    for _ in range(600):  # wait up to 1h for the training binary to start
        run_id = _read_run_id()
        if run_id: break
        time.sleep(6)
    if not run_id:
        print("system monitor: training never started (no run id found)")
        return
    run = wandb.init(project=project, group=group, name="system-monitor", id=run_id, resume=True)
    print(f"system monitor attached to run {run_id}")
    while True:
        d = {}
        g, c = _nvidia(), _cpu()
        if g: d.update({("sys/" + k): v for k, v in g.items()})
        if c: d.update({("sys/" + k): v for k, v in c.items()})
        if d: run.log(d)
        time.sleep(10)

t = threading.Thread(target=sys_monitor, args=("Carnage", "Phase 1"), daemon=True)
t.start()
print("system monitor thread started")

In [ ]:
# cell - 11
# TRAINING SUPERVISOR
# - Auto-selects device: CUDA if available, otherwise CPU (slow but runnable).
# - Starts the 24/7 backup daemon + a live log tailer, then launches training.
# - The tailer streams the binary's real-time output (steps/timesteps/reports) into THIS cell.
# - If the binary crashes, it auto-restarts with backoff, resuming from the latest checkpoint.
# - W&B group/run come from env: CARNAGE_WANDB_GROUP, CARNAGE_WANDB_RUN.
# - Games: CARNAGE_GAMES env override, default 1024.
import os, sys, subprocess, torch, time, threading, shutil, glob

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("WARNING: no GPU detected - training on CPU (will be slow)", flush=True)

BUILD = "/content/Carnage-V1/build"
LOCAL_CKPT = os.path.join(BUILD, "checkpoints")
DRIVE_CKPT = "/content/drive/MyDrive/Carnage-V1/checkpoints"
GAMES = int(os.environ.get("CARNAGE_GAMES", "1024"))

# ---- 24/7 backup daemon (guarded: one instance per kernel) ----
def _ts(d):
    try: return int(os.path.basename(d))
    except Exception: return -1

def _backup_once():
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    local = sorted([d for d in glob.glob(os.path.join(LOCAL_CKPT, "*")) if os.path.isdir(d)], key=_ts)
    drive = set(os.path.basename(d) for d in glob.glob(os.path.join(DRIVE_CKPT, "*")))
    for d in local:
        name = os.path.basename(d)
        if name in drive:
            continue
        tmp = os.path.join(DRIVE_CKPT, name + ".tmp")
        try:
            print(f"[backup] uploading checkpoint {name} ...", flush=True)
            shutil.copytree(d, tmp)
            shutil.move(tmp, os.path.join(DRIVE_CKPT, name))
            print(f"[backup] {name} uploaded", flush=True)
        except Exception as e:
            print("[backup] err:", e, flush=True)
            shutil.rmtree(tmp, ignore_errors=True)

def _backup_loop():
    while True:
        try:
            if os.path.isdir("/content/drive"):
                _backup_once()
            else:
                print("[backup] drive not mounted, retrying in 60s", flush=True)
        except Exception as e:
            print("[backup] loop err:", e, flush=True)
        time.sleep(60)

if "_CARNAGE_BACKUP_DAEMON_" not in globals():
    globals()["_CARNAGE_BACKUP_DAEMON_"] = True
    threading.Thread(target=_backup_loop, daemon=True).start()
    print("[backup] 24/7 daemon started (new checkpoints -> Drive within 60s)", flush=True)

# ---- Live log tailer: streams train.log (binary output + history) into this cell ----
_tailer_lock = {"run": True}
if "_CARNAGE_TAILER_" not in globals():
    globals()["_CARNAGE_TAILER_"] = True
    def _tail_loop():
        path = os.path.join(BUILD, "train.log")
        while not os.path.exists(path):
            time.sleep(1)
        while True:
            try:
                with open(path, "r", errors="ignore") as f:
                    while True:
                        data = f.read()
                        if data:
                            sys.stdout.write(data)
                            sys.stdout.flush()
                        else:
                            if not _tailer_lock.get("run", True):
                                return
                            time.sleep(0.5)
            except SystemExit:
                return
            except Exception:
                time.sleep(1)
    threading.Thread(target=_tail_loop, daemon=True).start()
    print("[tailer] streaming binary output to this cell...", flush=True)

# ---- Training supervisor ----
os.chdir(BUILD)
os.makedirs(LOCAL_CKPT, exist_ok=True)

BASE_CMD = ["stdbuf", "-oL", "-eL", "./Carnage", "/content/Carnage-V1/collision_meshes",
            "--device", DEVICE, "--games", str(GAMES), "--save-dir", "checkpoints", "--wandb", "Carnage"]
print("BASE CMD:", " ".join(BASE_CMD), flush=True)
print("Games:", GAMES, flush=True)

backoff = 10
attempt = 0
with open("train.log", "a") as log:
    while True:
        attempt += 1
        t0 = time.time()
        print(f"[supervisor] launch #{attempt}: {' '.join(BASE_CMD[:6])} ...", flush=True)
        proc = subprocess.Popen(BASE_CMD, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        rc = proc.wait()
        elapsed = time.time() - t0
        if rc == 0:
            print("[supervisor] clean exit (rc=0) - stopping supervisor.", flush=True)
            break
        print(f"[supervisor] crashed rc={rc} after {elapsed:.0f}s - restarting in {backoff}s", flush=True)
        time.sleep(backoff)
        if elapsed < 60:
            backoff = min(backoff * 2, 600)
        else:
            backoff = 10
print("supervisor exited.")

In [ ]:
# cell - 12
# STATUS: drive mounted? binary running? checkpoints present?
import os, subprocess, glob

print("drive mounted:", os.path.isdir("/content/drive"))
if os.path.isdir("/content/drive"):
    ck = "/content/drive/MyDrive/Carnage-V1/checkpoints"
    print("drive ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(ck + "/*")) if os.path.isdir(ck) else "none yet")

LOCAL = "/content/Carnage-V1/build/checkpoints"
print("local ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(LOCAL + "/*")) if os.path.isdir(LOCAL) else "none yet")

r = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
print("Carnage processes:", r.stdout.strip() if r.stdout.strip() else "none")

In [ ]:
# cell - 13
# STOP training cleanly: SIGTERM (binary saves a checkpoint then exits), SIGKILL fallback.
import subprocess, time
r = subprocess.run(["pkill", "-TERM", "-f", "Carnage"], capture_output=True, text=True)
print("SIGTERM sent:", r.returncode == 0)
time.sleep(20)
r2 = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
alive = r2.stdout.strip()
if alive:
    print("still alive, force killing:", alive)
    subprocess.run(["pkill", "-9", "-f", "Carnage"], capture_output=True)
    time.sleep(3)
r3 = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
print("remaining:", r3.stdout.strip() if r3.stdout.strip() else "none")